# S&P 500 Stock Market Performance and Business Intelligence - Track A (Data Analytics)
**AnalystLab Africa - Experience Lab Internship - Week 6** &nbsp;|&nbsp; Dataset: `S&P 500 Stock Prices 2014-2017.csv`

**Business problem.** Financial researchers need to understand how individual stocks behaved between 2014 and 2017: which delivered the highest
returns, which were most volatile, how trading activity varied and how volume relates to price moves. Raw prices alone mislead
(a $2,000 stock is not "better" than a $50 stock), so the analysis is built on **percentage returns, volatility and volume**, validated independently.

**Intended users:** equity research analysts and market-intelligence teams. **Decisions supported:** which stocks to study further, how to compare
risk/return, where data needs correcting before reporting. **Important:** all results are *historical (2014-2017)*; they are not current market data and not investment advice.

**Success criteria** (Task 3 of the common requirements)
1. Key figures reconcile with an independent pure-Python recomputation from the raw CSV (target: 100% of checks pass).
2. Every ranking uses percentage returns/volatility on a comparable (full-period) stock set, with data-quality exclusions documented.
3. The dashboards (Power BI + Plotly) expose the 4 required pages with symbol and date filters, and their filtered results match the source.
4. At least 5 findings and 3 recommendations, each with supporting evidence.

In [1]:
import sys, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "../02_Scripts")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import sp500_utils as u
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid")
raw = pd.read_csv(u.RAW_CSV, encoding="utf-8-sig")      # original file, never modified
print("Original file loaded:", raw.shape)

Original file loaded: (497472, 7)


## 1. Inspect the data (Task 1)

In [2]:
print("Columns :", list(raw.columns))
print(raw.dtypes.to_string())
print("\nFirst rows:"); print(raw.head(3).to_string(index=False))
df0 = u.load_raw()                                       # only change: date text (M/D/YYYY) -> datetime
print("\nRecords:", len(df0), "| Symbols:", df0.symbol.nunique(), "| Trading dates:", df0.date.nunique())
print("Date range:", df0.date.min().date(), "to", df0.date.max().date())
print("\nMissing values:\n", raw.isna().sum().to_string())
print("\nDuplicate symbol-date pairs:", df0.duplicated(["symbol","date"]).sum())
print("Zero-volume rows:", (df0.volume == 0).sum())
print("File sorted by symbol/date already?", df0.equals(df0.sort_values(["symbol","date"]).reset_index(drop=True)))

Columns : ['symbol', 'date', 'open', 'high', 'low', 'close', 'volume']
symbol        str
date          str
open      float64
high      float64
low       float64
close     float64
volume      int64

First rows:
symbol     date     open     high      low    close   volume
   AAL 1/2/2014  25.0700  25.8200  25.0600  25.3600  8998943
  AAPL 1/2/2014  79.3828  79.5756  78.8601  79.0185 58791957
   AAP 1/2/2014 110.3600 111.8800 109.2900 109.7400   542711

Records: 497472 | Symbols: 505 | Trading dates: 1007
Date range: 2014-01-02 to 2017-12-29

Missing values:
 symbol     0
date       0
open      11
high       8
low        8
close      0
volume     0

Duplicate symbol-date pairs: 0
Zero-volume rows: 4
File sorted by symbol/date already? False


In [3]:
print("Rows with missing open/high/low:"); print(df0[df0[["open","high","low"]].isna().any(axis=1)].to_string(index=False))
print("\nZero-volume rows:"); print(df0[df0.volume == 0].to_string(index=False))

Rows with missing open/high/low:
symbol       date  open  high   low    close  volume
  VRTX 2015-05-12   NaN   NaN   NaN 124.0800  569747
  REGN 2015-06-09   NaN   NaN   NaN 526.0900   12135
   WRK 2015-06-26   NaN   NaN   NaN  61.9000     100
   DHR 2015-07-17   NaN 88.76 88.24  88.7200 2056819
    ES 2015-07-17   NaN 48.49 47.85  47.9200 1246786
     O 2015-07-17   NaN 47.31 46.83  46.9900 1229513
   DHR 2016-01-12   NaN   NaN   NaN  88.5500       0
     O 2016-01-12   NaN   NaN   NaN  52.4300       0
    UA 2016-04-07   NaN   NaN   NaN  41.5600       0
   FTV 2016-07-01   NaN   NaN   NaN  49.5400       0
   BHF 2017-07-26   NaN   NaN   NaN  69.0842       3

Zero-volume rows:
symbol       date  open  high  low  close  volume
   DHR 2016-01-12   NaN   NaN  NaN  88.55       0
     O 2016-01-12   NaN   NaN  NaN  52.43       0
    UA 2016-04-07   NaN   NaN  NaN  41.56       0
   FTV 2016-07-01   NaN   NaN  NaN  49.54       0


In [4]:
cnt = df0.groupby("symbol").agg(first=("date","min"), last=("date","max"), days=("date","count"))
inc = cnt[cnt.days < df0.date.nunique()].sort_values("days")
print(f"{len(inc)} symbols have fewer than {df0.date.nunique()} records. Examples (shortest histories):")
print(inc.head(8).to_string())
print("\nHighest close :", df0.loc[df0.close.idxmax(), ["symbol","date","close"]].tolist())
print("Lowest close  :", df0.loc[df0.close.idxmin(), ["symbol","date","close"]].tolist())
print("Highest volume:", df0.loc[df0.volume.idxmax(), ["symbol","date","volume"]].tolist())

26 symbols have fewer than 1007 records. Examples (shortest histories):
            first       last  days
symbol                            
APTV   2017-12-05 2017-12-29    18
DWDP   2017-09-01 2017-12-29    83
BHF    2017-07-17 2017-12-29   117
BHGE   2017-07-03 2017-12-29   126
DXC    2017-04-03 2017-12-29   189
HLT    2017-01-04 2017-12-29   250
EVHC   2016-12-02 2017-12-29   271
FTV    2016-07-01 2017-12-29   378

Highest close : ['PCLN', Timestamp('2017-08-08 00:00:00'), np.float64(2049.0)]
Lowest close  : ['CHK', Timestamp('2016-02-12 00:00:00'), np.float64(1.59)]
Highest volume: ['VZ', Timestamp('2014-02-24 00:00:00'), np.int64(618237630)]


**Data-quality findings.** All figures in the assignment brief were confirmed: 11/8/8 missing open/high/low, no duplicates, 4 zero-volume rows (these also have blank O/H/L but a valid close),
and 26 symbols with incomplete histories (22 start late/end early). Further issues found: dates are text in `M/D/YYYY`; the file is not sorted by symbol-date; 9 rows have logically inconsistent OHLC values;
and a screen of every daily move above 25% (39 cases) shows some drops are almost certainly **spin-offs/splits/special dividends** (e.g. EBAY -57% when PayPal was spun off) because the file has no adjusted close.

## 2. Cleaning and preparation (Task 2)

In [5]:
df = u.prepare(df0)
new_cols = [c for c in df.columns if c not in raw.columns]
print("Engineered fields:", new_cols)
print("\nNon-market events excluded from return statistics (analyst-judgement list; verify with vendor):")
for (s, d), note in u.NON_MARKET_EVENTS.items(): print(f"  {s:6s} {d}  {note}")
print("\nLarge moves >25% retained as genuine market moves:", int(df.large_move.sum()))
print("OHLC-inconsistent rows flagged:", int(df.ohlc_inconsistent.sum()))
print(df[df.symbol == "AAPL"][["date","close","daily_change","daily_return_pct","daily_range","ma_5","ma_20","ma_50"]].iloc[[0,1,5,25,60]].to_string(index=False))

Engineered fields: ['year', 'quarter', 'month', 'month_name', 'year_month', 'year_quarter', 'missing_ohl', 'zero_volume', 'ohlc_inconsistent', 'daily_change', 'daily_return_pct', 'extreme_move', 'large_move', 'daily_return_clean_pct', 'abs_return_pct', 'cum_return_pct', 'return_bin', 'daily_range', 'daily_range_pct', 'ma_5', 'ma_20', 'ma_50', 'rolling_vol_20']

Non-market events excluded from return statistics (analyst-judgement list; verify with vendor):
  NI     2015-07-02  Probable spin-off (Columbia Pipeline Group)
  EBAY   2015-07-20  Probable spin-off (PayPal)
  DISCA  2014-08-07  Probable 2-for-1 split (stock dividend)
  DISCK  2014-08-07  Probable 2-for-1 split (stock dividend)
  BAX    2015-07-01  Probable spin-off (Baxalta)
  A      2014-11-03  Probable spin-off (Keysight Technologies)
  ARNC   2016-11-01  Probable separation (Alcoa Corp / Arconic)
  YUM    2016-11-01  Probable spin-off (Yum China)
  CAG    2016-11-10  Probable spin-off (Lamb Weston)
  HPE    2017-04-03  Prob

**Decisions.** (1) Missing O/H/L are left blank - never zero-filled; (2) zero-volume rows stay (valid close) but are excluded from volume statistics;
(3) every calculation is per symbol on date-sorted data; moving averages need a full window (no partial windows, no look-ahead);
(4) the 13 flagged non-market days are excluded from average-return / volatility and count as 0% in the event-adjusted cumulative return.

In [6]:
summ = u.symbol_summary(df)
full = summ[summ.history_status == "Full period"]
print("Symbols with a full-period history (used for rankings):", len(full), "of", len(summ))
SEL = u.SELECTED; print("Selected stocks:", SEL)

Symbols with a full-period history (used for rankings): 483 of 505
Selected stocks: ['AAPL', 'MSFT', 'AMZN', 'GOOGL', 'JPM', 'XOM', 'JNJ', 'NFLX', 'NVDA', 'WMT']


## 3. Market performance analysis (Task 3 - all 8 questions)

### Q1. Which stocks recorded the highest (and lowest) closing prices?
*Method: maximum / minimum `close` per symbol.*

In [7]:
print(summ.nlargest(8,"max_close")[["symbol","max_close","max_close_date"]].to_string(index=False))
print(); print(summ.nsmallest(5,"min_close")[["symbol","min_close","min_close_date"]].to_string(index=False))

symbol  max_close max_close_date
  PCLN    2049.00     2017-08-08
  AMZN    1195.83     2017-11-27
 GOOGL    1085.09     2017-12-18
  GOOG    1077.14     2017-12-18
   AZO     815.98     2016-07-28
   CMG     757.77     2015-08-05
   MTD     689.11     2017-11-01
  REGN     592.40     2015-08-05

symbol  min_close min_close_date
   CHK       1.59     2016-02-12
   AMD       1.62     2015-07-27
   FCX       3.74     2016-01-13
   MRO       6.73     2016-02-19
    RF       7.08     2016-02-11


**Interpretation.** PCLN (Priceline) peaked at $2,049 and AMZN at $1,196, while CHK touched $1.59 and AMD $1.62 - but price *level* says nothing about performance; it mainly reflects share count/split history. Use returns (Q2).

### Q2. Which stocks recorded the highest and lowest percentage returns?
*Method: (last close / first close - 1) x 100, event-adjusted; full-period symbols only.*

In [8]:
cols = ["symbol","first_close","last_close","raw_period_return_pct","adj_period_return_pct"]
print("Top 10:"); print(full.nlargest(10,"adj_period_return_pct")[cols].round(1).to_string(index=False))
print("\nBottom 10:"); print(full.nsmallest(10,"adj_period_return_pct")[cols].round(1).to_string(index=False))
print("\nMedian full-history return: %.1f%% | share of stocks with positive return: %.1f%%" % (full.adj_period_return_pct.median(), (full.adj_period_return_pct>0).mean()*100))

Top 10:
symbol  first_close  last_close  raw_period_return_pct  adj_period_return_pct
  NVDA         15.9       193.5                 1120.1                 1120.1
  AVGO         52.6       256.9                  387.9                  387.9
    EA         22.8       105.1                  360.2                  360.2
  ALGN         57.0       222.2                  290.1                  290.1
  NFLX         51.8       192.0                  270.4                  270.4
  ATVI         18.1        63.3                  250.4                  250.4
   LUV         18.9        65.4                  246.7                  246.7
  SWKS         27.4        95.0                  246.5                  246.5
    EW         33.0       112.7                  242.0                  242.0
   CNC         29.5       100.9                  241.7                  241.7

Bottom 10:
symbol  first_close  last_close  raw_period_return_pct  adj_period_return_pct
   CHK         26.6         4.0             

**Interpretation.** NVDA (+1,120%), AVGO (+388%) and EA (+360%) led; the weakest were mostly energy names (CHK -85%, RRC -79%, NBL, NOV) hit by the 2014-16 oil-price fall. The typical stock gained ~49% and ~81% finished higher. Returns exclude dividends (price return only).

### Q3. Which stocks experienced the greatest price volatility?
*Method: standard deviation of clean daily % returns; annualised x sqrt(252).*

In [9]:
v = full.sort_values("annualised_volatility_pct", ascending=False)[["symbol","daily_volatility_pct","annualised_volatility_pct","adj_period_return_pct"]]
print("Most volatile:"); print(v.head(8).round(2).to_string(index=False))
print("\nLeast volatile:"); print(v.tail(5).iloc[::-1].round(2).to_string(index=False))

Most volatile:
symbol  daily_volatility_pct  annualised_volatility_pct  adj_period_return_pct
   CHK                  4.55                      72.25                 -85.12
   AMD                  3.91                      62.07                 160.25
   FCX                  3.70                      58.72                 -49.61
   WMB                  3.23                      51.35                 -20.35
   MRO                  3.22                      51.06                 -51.50
   RRC                  2.91                      46.15                 -79.13
  INCY                  2.86                      45.34                  84.87
  VRTX                  2.82                      44.76                 102.98

Least volatile:
symbol  daily_volatility_pct  annualised_volatility_pct  adj_period_return_pct
   PEP                  0.83                      13.15                  46.07
    PG                  0.85                      13.50                  14.08
   RSG              

**Interpretation.** Energy/mining and high-growth names (CHK 72%, AMD 62%, FCX 59%, WMB 51% annualised) were 4-5x as volatile as defensive staples (PEP, PG, KO, WM ~13-14%).

### Q4. Which stocks recorded the highest trading volumes?
*Method: mean daily volume (zero-volume rows excluded) and single-day maximum.*

In [10]:
print(summ.nlargest(8,"avg_volume")[["symbol","avg_volume","max_volume"]].round(0).to_string(index=False))
print(); print(df.nlargest(5,"volume")[["symbol","date","close","volume"]].to_string(index=False))

symbol  avg_volume  max_volume
   BAC  89362904.0   375088650
  AAPL  45169571.0   266833581
    GE  41443943.0   431332632
   AMD  33289509.0   268336455
     F  32914301.0   142168574
  MSFT  30712613.0   202526536
    FB  29227824.0   150438699
    MU  27962814.0   149010345

symbol       date  close    volume
    VZ 2014-02-24  46.23 618237630
    GE 2015-11-17  30.32 431332632
   BAC 2016-02-11  11.16 375088650
    GE 2015-04-10  28.51 352701949
   BAC 2014-04-28  14.95 344935158


**Interpretation.** BAC (~89M shares/day), AAPL (~45M) and GE (~41M) were the most actively traded by share count. Volume counts *shares*, not money - low-priced stocks trade more shares.

### Q5. How did the selected stocks perform between 2014 and 2017?

**Interpretation.** All but XOM finished above their 2014 start. 2015 was the standout year for AMZN (+118%) and NFLX (+134%) while WMT fell 29%; NVDA's +224% in 2016 drove its +1,120% total.

### Q6. How did stock prices vary across months and years?
*Method: per-stock compounded monthly return, averaged equally across 483 full-history stocks.*

**Interpretation.** Aug-2015 (-5.5%) and Jan-2016 (-5.5%) were the weakest months; Oct-2015 (+7.2%) and Mar-2016 (+7.9%) the strongest. Feb/Oct/Nov averaged best and Jan/Aug/Sep worst, but with only four observations per month this is **not evidence of reliable seasonality**.

### Q7. Which stocks experienced the largest daily price changes?

In [13]:
ok = df[~df.extreme_move]
print("Largest daily $ changes:"); print(ok.assign(a=ok.daily_change.abs()).nlargest(5,"a")[["symbol","date","close","daily_change","daily_return_pct"]].round(2).to_string(index=False))
print("\nLargest % gains:"); print(ok.nlargest(5,"daily_return_pct")[["symbol","date","close","daily_return_pct"]].round(1).to_string(index=False))
print("\nLargest % losses:"); print(ok.nsmallest(5,"daily_return_pct")[["symbol","date","close","daily_return_pct"]].round(1).to_string(index=False))

Largest daily $ changes:
symbol       date   close  daily_change  daily_return_pct
  PCLN 2017-11-07 1645.72       -257.28            -13.52
  PCLN 2016-06-24 1232.14       -158.06            -11.37
  PCLN 2017-08-09 1906.80       -142.20             -6.94
  PCLN 2015-11-09 1311.15       -138.75             -9.57
  AMZN 2017-10-27 1100.95        128.52             13.22

Largest % gains:
symbol       date  close  daily_return_pct
   AMD 2016-04-22    4.0              52.3
  VRTX 2014-06-24   93.5              40.4
   NWL 2017-09-15   43.2              40.2
   MRO 2017-09-14   16.0              34.5
   CHK 2016-04-12    6.0              34.4

Largest % losses:
symbol       date  close  daily_return_pct
   WMB 2016-02-08   11.2             -34.8
  EVHC 2017-11-01   28.0             -34.2
   CHK 2016-02-08    2.0             -33.3
   SIG 2017-11-21   52.8             -30.4
   NWL 2017-09-14   30.8             -30.3


**Interpretation.** Dollar changes are dominated by the highest-priced stock (PCLN), so percentage moves are the fairer yardstick: AMD +52% (22 Apr 2016), WMB -35% (8 Feb 2016) and EVHC -34% (1 Nov 2017) were the biggest genuine one-day moves.

### Q8. How does trading activity differ across the selected stocks?

In [14]:
q8 = pd.read_csv(u.OUT / "q8_selected_volume_stats.csv")
print(q8.round(2).to_string(index=False))

symbol  avg_volume  median_volume  std_volume  min_volume  max_volume  volume_cv
  AAPL 45169571.17     39232609.0 24902326.42    11475922   266833581       0.55
  MSFT 30712613.15     27343581.0 15713880.24     7425603   202526536       0.51
  AMZN  3880124.57      3340542.0  2227803.04     1092970    23856060       0.57
 GOOGL  2107183.87      1735469.0  1226332.32      521141    12858136       0.58
   JPM 15680002.05     14278117.0  6164955.05     3680043    56192320       0.39
   XOM 12298770.50     11149690.0  4836882.56     3851283    44704931       0.39
   JNJ  7298176.87      6730605.0  2785257.32     2353779    25835156       0.38
  NFLX 14373715.26     11571389.0 11046729.06     1616260   104500746       0.77
  NVDA 10761294.53      8547710.0  7656878.57     1141128    92323196       0.71
   WMT  8455316.68      7318362.0  4831361.07     2483121    80898117       0.57


**Interpretation.** AAPL and MSFT trade tens of millions of shares daily; AMZN and GOOGL only 2-4 million because of their high share prices. NFLX (CV 0.77) and NVDA (0.71) had the most erratic volume.

## 4. Comparative stock analysis (Task 4)

In [15]:
comp = pd.read_csv(u.OUT / "task4_comparative_stock_table.csv")
show = ["symbol","total_return_pct","mean_daily_return_pct","annualised_volatility_pct","return_per_unit_risk","highest_close","lowest_close","avg_volume","pct_days_close_above_ma50"]
print(comp[show].round(2).to_string(index=False))
print("\nMoving-average position at 29 Dec 2017:"); print(comp[["symbol","latest_ma5","latest_ma20","latest_ma50","ma20_vs_ma50_at_end"]].round(2).to_string(index=False))

symbol  total_return_pct  mean_daily_return_pct  annualised_volatility_pct  return_per_unit_risk  highest_close  lowest_close  avg_volume  pct_days_close_above_ma50
  AAPL            114.16                   0.09                      22.71                  0.95         176.42         71.40 45169571.17                      66.70
  MSFT            130.19                   0.09                      21.74                  1.07          86.85         34.98 30712613.15                      73.38
  AMZN            193.86                   0.12                      29.60                  1.06        1195.83        286.95  3880124.57                      68.48
 GOOGL             89.08                   0.07                      21.94                  0.84        1085.09        497.06  2107183.87                      60.86
   JPM             83.71                   0.07                      20.55                  0.84         107.83         53.07 15680002.05                      70.04
   XOM    

**Interpretation.** On a risk-adjusted basis NVDA (return per unit of risk 1.85) led, followed by MSFT (1.07), AMZN (1.06) and NFLX (0.99); XOM was the only negative. Daily returns of the selected stocks are positively but moderately correlated (0.10-0.57;
GOOGL-AMZN 0.57, GOOGL-MSFT 0.55 highest; NVDA-WMT 0.10 lowest). Volume and the *size* of the daily move are positively related for every stock (Spearman 0.41-0.51): heavier trading accompanies bigger moves (association, not causation).

## 5. Interactive dashboards (Task 5)

Two interactive dashboards were built on the prepared data:
* **Power BI** - open `04_PowerBI/SP500_Dashboard.pbip` (4 pages: Market Overview, Stock Price Performance, Returns and Volatility, Trading Volume Analysis; symbol and date slicers). See `04_PowerBI/PowerBI_Setup_and_Manual_Build_Guide.md`.
* **Plotly** - open `05_Plotly_Dashboard/SP500_Interactive_Dashboard.html` (same 4 pages with symbol / date filters). The cell below shows the Plotly API version of one chart when `plotly` is installed.

In [18]:
try:
    import plotly.express as px
    d = df[df.symbol.isin(SEL[:5])]
    px.line(d, x="date", y="close", color="symbol", title="Closing price - selected stocks").show()
except ImportError:
    print("plotly is not installed in this environment; the interactive Plotly dashboard is in 05_Plotly_Dashboard/ (HTML, loads Plotly.js from the CDN).")

plotly is not installed in this environment; the interactive Plotly dashboard is in 05_Plotly_Dashboard/ (HTML, loads Plotly.js from the CDN).


## 6. Validation (Task 6)

In [19]:
val = pd.read_csv(u.VAL / "validation_results.csv")
print(val.result.value_counts().to_string())
print(val[["check","independent_value","pipeline_value","result"]].head(14).to_string(index=False))
print("... full table in 08_Validation/validation_results.csv")

result
PASS    41
                                                 check  independent_value        pipeline_value result
                  V1a Total records (raw vs processed)             497472                497472   PASS
                                    V1b Unique symbols                505                   505   PASS
                     V1c Power BI Prices.csv row count             497472                497472   PASS
                            V1d Distinct trading dates               1007                  1007   PASS
V2 AAPL daily return: max |difference| over 1,006 days                0.0 4.971527634012318e-10   PASS
   V2b AAPL first-day return is blank (no prior close)               True                  True   PASS
     V3 AAPL MA5: count of values and max |difference|                0.0 2.842170943040401e-14   PASS
    V3 AAPL MA20: count of values and max |difference|                0.0 2.842170943040401e-14   PASS
    V3 AAPL MA50: count of values and max |difference| 

All 41 independent checks passed (record counts, AAPL daily returns, MA5/20/50, total returns, average closes, volatility, extremes, filter test and Power BI table reconciliation).

## 7. Business insights and recommendations (Task 7)

### Findings
1. **Leaders and laggards were sector-driven.** Technology/growth names led (NVDA +1,120%, AVGO +388%, EA +360%); energy led the losers (CHK -85%, RRC -79%). Median stock +49%, 81% positive.
2. **Price level is not performance.** PCLN's $2,049 peak and CHK's $1.59 low are share-price levels; rankings must use % returns (NVDA +1,120% vs PCLN far lower).
3. **Risk varies 5x across stocks.** CHK/AMD/FCX volatility of 59-72% vs ~13-14% for PEP/PG/KO; among selected stocks NFLX (42%) and NVDA (38%) were ~3x as volatile as JNJ (14%).
4. **Best risk-adjusted performer was NVDA** (1.85 return per unit of risk), well ahead of MSFT/AMZN (~1.06); XOM destroyed value (-16%) with moderate volatility.
5. **Volume rises with price-move size.** Spearman 0.41-0.51 (p<0.001) between volume and |daily return| for all 10 selected stocks.
6. **Diversification is visible in the data.** Selected-stock return correlations range 0.10-0.57; WMT/NVDA (0.10) and JNJ/NFLX (0.14) move most independently, GOOGL/AMZN (0.57) most together.
7. **Data limitations materially change results.** 13 non-market days (e.g. EBAY -57%, NI -63%) would corrupt returns/volatility if left in; 22 symbols have partial histories and cannot be ranked against full-period ones.

### Recommendations
| # | Recommendation | Evidence | How stakeholders use it |
|---|---|---|---|
| 1 | Obtain an adjusted-close/corporate-action feed (or verify flagged events) before publishing long-term returns | 13 flagged days with -35% to -63% one-day drops; 26 incomplete histories | Research team signs off return tables only after corporate actions are confirmed |
| 2 | Compare stocks with % return, volatility and return-per-risk together, never raw price | PCLN/CHK price extremes vs NVDA/CHK return extremes; NVDA 1.85 vs XOM -0.16 return per risk | Analysts shortlist stocks on risk-adjusted return and size positions by volatility |
| 3 | Use volume spikes as a screening flag for unusual price moves, and pair low-correlation stocks for diversification | Spearman 0.41-0.51 volume vs move size; correlations 0.10-0.57 | Monitoring desk flags days with heavy volume and large moves; portfolio researchers test WMT/NVDA-type pairs |

*Historical performance does not guarantee future returns. No predictive claims are made in this analysis.*